# Additional End of week Exercise - week 2

Now use everything you've learned from Week 2 to build a full prototype for the technical question/answerer you built in Week 1 Exercise.

This should include a Gradio UI, streaming, use of the system prompt to add expertise, and the ability to switch between models. Bonus points if you can demonstrate use of a tool!

If you feel bold, see if you can add audio input so you can talk to it, and have it respond with audio. ChatGPT or Claude can help you, or email me if you have questions.

I will publish a full solution here soon - unless someone beats me to it...

There are so many commercial applications for this, from a language tutor, to a company onboarding solution, to a companion AI to a course (like this one!) I can't wait to see your results.

In [39]:
import os
import json
from dotenv import load_dotenv
from openai import OpenAI
from IPython.display import Markdown, display
import gradio as gr 

In [9]:
MODEL_GPT = 'gpt-4o-mini'
MODEL_LLAMA = 'llama3.2'
BASE_URL = 'https://openrouter.ai/api/v1'
OLLAMA_BASE_URL = 'http://localhost:11434/v1'

In [ ]:
load_dotenv(override=True)
api_key = os.getenv('OPEN_ROUTER_API_KEY')

# Check the key

if not api_key:
    print("No API key was found - please head over to the troubleshooting notebook in this folder to identify & fix!")
elif not api_key.startswith("sk-or-"):
    print("An API key was found, but it doesn't start sk-or-; please check you're using the right key - see troubleshooting notebook")
elif api_key.strip() != api_key:
    print("An API key was found, but it looks like it might have space or tab characters at the start or end - please remove them - see troubleshooting notebook")
else:
    print("API key found and looks good so far!")

In [ ]:
system_prompt ="""
        You are a professional AI coding tutor.
        When given a programming question, provide a comprehensive explanation that is clear,
        and designed for learners seeking a deep understanding.
        Break down concepts, highlight key ideas, use relevant code examples, and point out common mistakes or misconceptions.
       """

user_prompt = """
        what is the highest acceptable column that can be indexed in the common sql database
        """


In [13]:
client = OpenAI(base_url=BASE_URL, api_key=api_key)


def gradio_response(user_message):
    """Wrapper: takes only the user message from Gradio and returns the model response."""
    response = client.chat.completions.create(
        model=MODEL_GPT, messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_message}
        ]
    )
    return response.choices[0].message.content or ""


In [19]:
force_dark_mode = """
function refresh() {
    const url = new URL(window.location);
    if (url.searchParams.get('__theme') !== 'dark') {
        url.searchParams.set('__theme', 'dark');
        window.location.href = url.href;
    }
}"""

In [ ]:
message_input = gr.Textbox(label="Your message:", info="Enter a message for GPT-4.1-mini", lines=7)
message_output = gr.Textbox(label="Response:", lines=10)

view = gr.Interface(
    fn=gradio_response,
    title="Master Interview Questions",
    inputs=[message_input],
    outputs=[message_output],
    examples=["Explain the Transformer achitecture to a layman", "Explain the Transformer achitecture to an aspiring AI Engineer"],
    flagging_mode="never", js=force_dark_mode
)
view.launch(inbrowser=True)

In [26]:
# GPT stream
def gpt_stream_response(user_prompt):
    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": user_prompt}
    ]

    stream = client.chat.completions.create(model=MODEL_GPT, messages=messages, stream=True)
    response = ""
    for chunk in stream:
        response += chunk.choices[0].delta.content or ""
        yield response


# OLLAMA stream
def ollama_stream_response(user_prompt):
    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": user_prompt}
    ]

    stream = client.chat.completions.create(model=MODEL_LLAMA, messages=messages, stream=True)
    response = ""
    for chunk in stream:
        response += chunk.choices[0].delta.content or ""
        yield response

In [ ]:
message_input = gr.Textbox(label="Your message:", info="Enter a message for GPT-4.1-mini", lines=7)
message_output = gr.Markdown(label="Response:")

view = gr.Interface(
    fn=gpt_stream_response,
    title="Master Interview Questions",
    inputs=[message_input],
    outputs=[message_output],
    examples=["Explain the Transformer achitecture to a layman", "Explain the Transformer achitecture to an aspiring AI Engineer"],
    flagging_mode="never",  js=force_dark_mode
)
view.launch(inbrowser=True)

In [21]:
# stream different model

def stream_model(prompt, model):
    if model=="GPT":
        result = gpt_stream_response(prompt)
    elif model=="OLLAMA":
        result = ollama_stream_response(prompt)
    else:
        raise ValueError("Unknown model")
    yield from result

In [ ]:
message_input = gr.Textbox(label="Your message:", info="Enter a message for the LLM", lines=7)
model_selector = gr.Dropdown(["GPT", "OLLAMA"], label="Select model", value="GPT")
message_output = gr.Markdown(label="Response:")

view = gr.Interface(
    fn=stream_model,
    title="Master Interview Questions", 
    inputs=[message_input, model_selector], 
    outputs=[message_output], 
    examples=[
            ["Explain the Transformer architecture to a layperson", "GPT"],
            ["Explain the Transformer architecture to an aspiring AI engineer", "OLLAMA"]
        ], 
    flagging_mode="never",  js=force_dark_mode
    )
view.launch(inbrowser=True)

In [ ]:
# Improvement on Master Interview Questions - A technical question/answer built in Week 1 by trying out interactive chat

def chat(message, history):
    history = [{"role":h["role"], "content":h["content"]} for h in history]
    messages = [{"role": "system", "content": system_prompt}] + history + [{"role": "user", "content": message}]
    stream = client.chat.completions.create(model=MODEL_GPT, messages=messages, stream=True)
    response = ""
    for chunk in stream:
        response += chunk.choices[0].delta.content or ''
        yield response

gr.ChatInterface(fn=chat, type="messages", js=force_dark_mode).launch(inbrowser=True)


In [ ]:
# Chat with tools

# Tool: return a short code example for a topic (fits the coding tutor use case)
sample_data = {
    "list comprehension": "squares = [x**2 for x in range(10)]",
    "decorator": "@functools.lru_cache(maxsize=128)\ndef expensive(n): ...",
    "context manager": "with open('f.txt') as f:\n    content = f.read()",
}

def get_code_example(topic):
    """Return a brief code example for the given programming topic."""
    print(f"Tool called for Example {topic}")
    key = topic.lower().strip()
    return sample_data.get(key, f"No example stored for '{topic}'. Try: list comprehension, decorator, context manager.")

In [37]:
get_code_example_function = {
    "name": "get_code_example",
    "description": "Get a short code example for a programming topic. Use when the user asks for an example or how to write something.",
    "parameters": {
        "type": "object",
        "properties": {
            "topic": {"type": "string", 
            "description": "The programming concept or topic (e.g. list comprehension, decorator)."
            },
        },
        "required": ["topic"],
        "additionalProperties": False,
    },
}

tools = [{"type": "function", "function": get_code_example_function}]


In [ ]:
def handle_tool_calls(message):
    responses = []
    for tool_call in message.tool_calls:
        if tool_call.function.name == "get_code_example":
            arguments = json.loads(tool_call.function.arguments)
            topic = arguments.get("topic", "")
            result = get_code_example(topic)
            responses.append({
                "role": "tool",
                "content": result,
                "tool_call_id": tool_call.id,
            })
    return responses

def chat_with_tools(message, history):
    history = [{"role": h["role"], "content": h["content"]} for h in history]
    messages = [{"role": "system", "content": system_prompt}] + history + [{"role": "user", "content": message}]
    response = client.chat.completions.create(model=MODEL_GPT, messages=messages, tools=tools)

    while response.choices[0].finish_reason == "tool_calls":
        message = response.choices[0].message
        responses = handle_tool_calls(message)
        messages.append(message)
        messages.extend(responses)
        response = client.chat.completions.create(model=MODEL_GPT, messages=messages, tools=tools)

    return response.choices[0].message.content or ""

gr.ChatInterface(fn=chat_with_tools, type="messages", js=force_dark_mode).launch(inbrowser=True)